# Advanced Topics

This notebook demonstrates three advanced tool patterns: dynamic tools (filtering tools by phase), human-in-the-loop approval via deferred tools, and the tool doctor for development-time analysis.

## Dynamic Tools

Dynamic tools reshape the agent's available toolset at each step based on context. Here, a `prepare_tools` function filters tools by a "phase" passed as the agent's dependency. In the "explore" phase, only read tools are exposed. In the "execute" phase, all tools become available.

In [ ]:
from pydantic_ai import RunContext
from pydantic_ai.tools import ToolDefinition

from agentic_patterns.core.agents import get_agent


def list_files(directory: str) -> list[str]:
    """List files in a directory."""
    return ["report.csv", "config.json", "data.parquet"]


def read_file(path: str) -> str:
    """Read the contents of a file."""
    return f"Contents of {path}: [sample data]"


def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    return f"Wrote {len(content)} chars to {path}"


def delete_file(path: str) -> str:
    """Delete a file permanently."""
    return f"Deleted {path}"


READ_TOOL_NAMES = {"list_files", "read_file"}


async def filter_by_phase(
    ctx: RunContext[str], tool_defs: list[ToolDefinition]
) -> list[ToolDefinition] | None:
    """In 'explore' phase, expose only read tools. Otherwise, expose all."""
    if ctx.deps == "explore":
        return [td for td in tool_defs if td.name in READ_TOOL_NAMES]
    return tool_defs


agent = get_agent(
    tools=[list_files, read_file, write_file, delete_file],
    prepare_tools=filter_by_phase,
    deps_type=str,
)

### Explore phase

In explore phase, the agent only sees `list_files` and `read_file`. It cannot delete or write.

In [ ]:
result = await agent.run(
    "List all files in /data and then delete config.json.",
    deps="explore",
)
print(result.output)

### Execute phase

In execute phase, all tools are available.

In [ ]:
result = await agent.run("Delete config.json.", deps="execute")
print(result.output)

## Human in the Loop (Deferred Tools)

Deferred tools separate tool selection from execution. A tool marked with `requires_approval=True` pauses the agent run and returns a `DeferredToolRequests` object. The caller reviews each request, approves or denies it, then resumes the run with `DeferredToolResults`.

In [ ]:
from pydantic_ai import DeferredToolRequests, DeferredToolResults, Tool

from agentic_patterns.core.agents import get_agent


def check_balance(account_id: str) -> str:
    """Check the current balance of a bank account."""
    return f"Account {account_id} balance: $5,230.00"


def transfer_funds(from_account: str, to_account: str, amount: float) -> str:
    """Transfer funds between two bank accounts."""
    return f"Transferred ${amount:.2f} from {from_account} to {to_account}"


agent = get_agent(
    tools=[check_balance, Tool(transfer_funds, requires_approval=True)],
    output_type=[str, DeferredToolRequests],
)

### First run: agent proposes a transfer but pauses for approval

In [ ]:
result = await agent.run(
    "Check the balance of ACC-123 and transfer $500 from ACC-123 to ACC-456."
)

assert isinstance(result.output, DeferredToolRequests)
requests = result.output
messages = result.all_messages()

print(f"Deferred approvals: {len(requests.approvals)}")
for call in requests.approvals:
    print(f"  {call.tool_name}({call.args}) [id={call.tool_call_id}]")

### Resume: approve the transfer and continue

In [ ]:
deferred_results = DeferredToolResults()
for call in requests.approvals:
    deferred_results.approvals[call.tool_call_id] = True

result = await agent.run(
    message_history=messages, deferred_tool_results=deferred_results
)
print(result.output)

To deny a tool call instead, use `ToolDenied`:

```python
deferred_results.approvals[call.tool_call_id] = ToolDenied("Transfer not allowed")
```

## Tool Doctor

The tool doctor analyzes tool definitions at development time and produces structured recommendations. It evaluates naming, documentation, type annotations, and argument clarity. Here we run it against three intentionally under-documented tools.

In [ ]:
from agentic_patterns.core.doctors.tool_doctor import tool_doctor


def process(data, flag=False):
    """Process data."""
    return data


def calc(x, y):
    return x + y


def fetch_and_transform(
    url: str, format: str = "json", retries: int = 3, timeout: float = 30.0
) -> dict:
    """Fetch data from URL."""
    return {"status": "ok"}

In [ ]:
recommendations = await tool_doctor([process, calc, fetch_and_transform])
for r in recommendations:
    print(r)
    print()